# 02 — Synthetic Rule Pool Generation

**Section:** Evaluation layer (PROMETHEE II, CART baseline, statistical testing, GSEA)
**Depends on:** `01_synthetic_data_foundation.ipynb` (`tests/fixtures/synthetic/`)
**Feeds into:** `03_promethee_ranking.ipynb` (not yet built)

## Recommended reading before this notebook

- **CONTRACTS.md §0.4** (this repo, `docs/CONTRACTS.md`) — the rule-pool schema every extractor
  must emit. This notebook exists to produce data that conforms to it.
- **Yedjour et al., GPRO_RX** — the GA + PROMETHEE pedagogical rule-extraction approach this
  project's chromosome encoding is modelled on (rule = one candidate solution, gene = one
  feature-partition choice).
- **Deb et al. (2002), "A fast and elitist multiobjective genetic algorithm: NSGA-II"**
  — for the non-dominated sorting / crowding-distance concepts `nsga.py` implements, even
  though this notebook doesn't run the full evolutionary loop (see §2 below for why).
- **DEAP documentation** (<https://deap.readthedocs.io>) — `common.py`'s `cxTwoPoint_partitions`
  wraps DEAP's built-in two-point crossover; worth skimming the `creator`/`toolbox`/`Individual`
  concepts if `nsga.py` itself is unfamiliar.
- **scikit-learn, `f1_score` docs** (`average="macro"`) — `metrics.fidelity_f1` is a direct
  call to this; macro-averaging matters here because sensitive/resistant classes may be
  imbalanced per drug.
- **Wu et al., DeepCCDS** — feature semantics (pathway activity is continuous; mutation status
  is binary), directly relevant to §3 below.

## What this notebook does, and why

Notebook 01 produced synthetic *predictions* (continuous LN_IC50 + discretised classes) —
what DeepCCDS would output. PROMETHEE II (notebook 03) doesn't rank predictions, though; it
ranks **rules**. Before a real GA or NSGA-II output exists, this notebook fabricates
schema-valid, GA/NSGA-II-*shaped* rule pools against notebook 01's synthetic labels, so the
PROMETHEE notebook has something real to run against.

**Important distinction, kept visible throughout this notebook rather than glossed over:**
this does **not** run the actual evolutionary search in `nsga.py` (non-dominated sorting,
crowding distance, 30 generations × 30 seeded runs). It generates a pool of *candidate* rules
directly, using the same shared building blocks the real algorithm uses
(`src.extraction.common.random_gene`/`decode_individual`/`resolve_genes`,
`src.common.metrics.evaluate_rule_mask`/`rule_class_preds`/`fidelity_f1`,
`src.common.schema.save_rule_pool`), then filters and sorts them by fidelity as a stand-in
for what a converged search would return.

This is a deliberate simplification, not a hidden one: every function called below is the
*real* shared contract code (same code GA/NSGA-II will use), so this notebook is a genuine
integration test of that code even though the search procedure itself is fabricated. Once
you're ready to compare fabricated vs. real search, `src/extraction/nsga.py` can be run
directly against this same synthetic data reshaped into the `artifacts/` layout it expects
(see the closing section) — worth doing once the pipeline is stable, to see how much the
real evolutionary search actually improves on random sampling.


In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "..")  # repo root, so `import src...` resolves

from src.common import metrics, schema
from src.common.config import load_config
from src.extraction.common import random_gene, decode_individual, resolve_genes

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

cfg = load_config("../configs/pipeline.yaml")
print(f"config_hash: {cfg.config_hash}")
print(f"discretisation thresholds: high_z={cfg.discretisation.high_z}, low_z={cfg.discretisation.low_z}")

FIXTURE_DIR = Path("../tests/fixtures/synthetic")
OUT_DIR = Path("../tests/fixtures/synthetic/rules")  # fabricated pools, NOT artifacts/rules/ —
# see 01's repo-structure note: synthetic stand-ins stay out of artifacts/ so nobody mistakes
# them for real GA/NSGA-II output once those branches land.


config_hash: a2070aedcc19
discretisation thresholds: high_z=1.0, low_z=-1.0


## 1. Load notebook 01's synthetic data

Two conditions (baseline, SAM) share the same cell-line features; only the predicted classes
differ. Rule extraction happens per condition, so both get their own rule pools below.


In [2]:
features_df = pd.read_csv(FIXTURE_DIR / "synthetic_features.csv", index_col="cell_line_id")
labels_df = pd.read_csv(FIXTURE_DIR / "synthetic_labels.csv", index_col="cell_line_id")
ground_truth = json.load(open(FIXTURE_DIR / "synthetic_ground_truth_meta.json"))

gene_cols = [c for c in features_df.columns if c.startswith("GENE_")]
pathway_cols = [c for c in features_df.columns if c.startswith("PATHWAY_")]

print(f"features: {features_df.shape}  ({len(gene_cols)} genes, {len(pathway_cols)} pathways)")
print(f"labels:   {labels_df.shape}")
print(f"true signal genes:    {list(ground_truth['true_signal_genes'].keys())}")
print(f"true signal pathways: {list(ground_truth['true_signal_pathways'].keys())}")
features_df.head(3)


features: (300, 59)  (20 genes, 38 pathways)
labels:   (300, 8)
true signal genes:    ['GENE_11', 'GENE_12', 'GENE_06', 'GENE_17']
true signal pathways: ['PATHWAY_30', 'PATHWAY_19', 'PATHWAY_25']


,cancer_type,GENE_01,GENE_02,GENE_03,GENE_04,GENE_05,GENE_06,GENE_07,GENE_08,GENE_09,...,PATHWAY_29,PATHWAY_30,PATHWAY_31,PATHWAY_32,PATHWAY_33,PATHWAY_34,PATHWAY_35,PATHWAY_36,PATHWAY_37,PATHWAY_38
cell_line_id,,,,,,,,,,,,,,,,,,,,,
SYN_CL_0001,CANCER_TYPE_1,0,0,0,1,0,0,0,1,1,...,-1.197054,1.989360,-0.096382,0.644514,-0.536869,-1.051928,-0.233881,1.101193,-0.520732,-0.870532
SYN_CL_0002,CANCER_TYPE_5,1,0,0,1,0,0,0,1,0,...,0.218819,-1.289335,-0.335039,-0.518845,0.464450,1.017022,0.991067,-0.842812,0.998468,-0.044085
SYN_CL_0003,CANCER_TYPE_4,1,0,0,1,0,1,0,0,1,...,1.285997,-0.210830,-0.141779,0.170046,0.891898,-1.470216,0.736056,-0.020496,-1.058637,-2.084606


## 2. Discretising features into rule-condition partitions

`schema.py`'s `VALID_PARTITIONS` fixes every rule condition to one of `{"Low", "Intermediate",
"High"}` — this is already-merged, shared code that GA and NSGA-II both validate against, so
changing it here isn't an option without breaking their branches too.

That's a clean fit for **pathway activity** (continuous, roughly standard-normal): z-score
thresholds from `pipeline.yaml`'s `discretisation` block (`high_z=1.0`, `low_z=-1.0`) give a
natural three-way split.

**Mutation status is binary (0/1)**, though, and doesn't naturally have three levels. Two
options were considered:

1. **Map 0/1 onto two of the three existing labels** (wild-type → `"Low"`, mutated →
   `"High"`; `"Intermediate"` simply never occurs for mutation columns).
2. Exclude mutation features from rule conditions entirely, ranking rules on pathway activity
   alone.

Option 1 is used below. Reasoning: the project's own decision record (session decision C2,
`COS700_Project_Orientation_and_Decisions.md`) already locked in *"pathway activity **plus**
mutation status"* as the rule-extraction feature space — excluding mutations here would
silently contradict that decision and lose a biologically important signal (driver-gene
mutation status is one of DeepCCDS's two interpretable input blocks). Mapping onto the
existing two-of-three labels respects the schema exactly as merged, needs no coordination
with Melisa/Sakhile's already-tested validation code, and reads naturally in a rule's IF-THEN
form (`IF TP53_mut = High AND ... THEN resistant` parses the same way as a pathway condition).


In [3]:
def discretise_pathway(series: pd.Series, high_z: float, low_z: float) -> pd.Series:
    '''Continuous pathway activity -> Low/Intermediate/High by z-score threshold.'''
    z = (series - series.mean()) / series.std()
    return pd.Series(
        np.select([z > high_z, z < low_z], ["High", "Low"], default="Intermediate"),
        index=series.index,
    )

def discretise_mutation(series: pd.Series) -> pd.Series:
    '''Binary mutation status -> Low (wild-type) / High (mutated). See markdown above for
    why Intermediate is never used here -- this is a deliberate 2-of-3 mapping, not a bug.'''
    return series.map({0: "Low", 1: "High"})

discretised_df = pd.DataFrame(index=features_df.index)
for col in pathway_cols:
    discretised_df[col] = discretise_pathway(features_df[col], cfg.discretisation.high_z, cfg.discretisation.low_z)
for col in gene_cols:
    discretised_df[col] = discretise_mutation(features_df[col])

feature_cols = gene_cols + pathway_cols  # fixed column order — chromosome position i <-> feature_cols[i]

print(discretised_df[pathway_cols[:2] + gene_cols[:2]].head())
print()
for col in pathway_cols[:2]:
    print(f"{col}: {discretised_df[col].value_counts().to_dict()}")
for col in gene_cols[:2]:
    print(f"{col}: {discretised_df[col].value_counts().to_dict()}  (Intermediate absent, as expected)")


                PATHWAY_01    PATHWAY_02 GENE_01 GENE_02
cell_line_id                                            
SYN_CL_0001            Low  Intermediate     Low     Low
SYN_CL_0002   Intermediate  Intermediate    High     Low
SYN_CL_0003   Intermediate  Intermediate    High     Low
SYN_CL_0004   Intermediate          High     Low     Low
SYN_CL_0005   Intermediate  Intermediate    High    High

PATHWAY_01: {'Intermediate': 200, 'Low': 52, 'High': 48}
PATHWAY_02: {'Intermediate': 207, 'High': 47, 'Low': 46}
GENE_01: {'Low': 186, 'High': 114}  (Intermediate absent, as expected)
GENE_02: {'Low': 168, 'High': 132}  (Intermediate absent, as expected)


## 3. `feature_map` — the gene-resolvability bridge

`schema.py`'s `validate_rule_pool(..., feature_map=...)` checks every rule's `genes` field
actually resolves to a real gene symbol via `feature_map.json` — this is what lets
Evaluation's GSEA step (notebook 05, not yet built) run directly off the `genes` field
without re-deriving it. Building a synthetic version now so this check is exercised for real
rather than skipped.


In [4]:
feature_map = {}
for col in gene_cols:
    feature_map[col] = {"gene_symbol": col.replace("GENE_", "GENE"), "partition_edges": None}
for col in pathway_cols:
    feature_map[col] = {"gene_symbol": col, "partition_edges": [cfg.discretisation.low_z, cfg.discretisation.high_z]}

print(f"feature_map entries: {len(feature_map)}")
print("sample:", {k: feature_map[k] for k in list(feature_map)[:2]})


feature_map entries: 58
sample: {'GENE_01': {'gene_symbol': 'GENE01', 'partition_edges': None}, 'GENE_02': {'gene_symbol': 'GENE02', 'partition_edges': None}}


## 4. Fabricating a rule pool

### 4.1 First attempt (shown failing deliberately, not deleted)

The obvious approach: draw every gene of a chromosome from `common.random_gene()` uniformly
— the exact same per-gene sampling function the real GA/NSGA-II initial population uses
(`nsga.py`'s `build_toolbox` registers `toolbox.register("attr_partition", random_gene)`
directly). Tried first, and worth showing why it fails rather than silently swapping it out.


In [5]:
MIN_COVERAGE = 0.02
N_CANDIDATES = 400   # random chromosomes drawn per pool — arbitrary but generous for a stand-in
TOP_K = 15            # how many survive into the fabricated "pool"

def fabricate_rule_pool_naive(condition: str, class_col: str, seed: int) -> int:
    '''First attempt, kept here deliberately rather than deleted: draws every gene from
    common.random_gene() uniformly, exactly as a real GA/NSGA-II initial population would.
    Included to show why it does not work as a fabrication strategy, not as the final version.'''
    import random
    random.seed(seed)
    n_features = len(feature_cols)
    survivors = 0
    for _ in range(N_CANDIDATES):
        individual = [random_gene() for _ in range(n_features)]
        conditions = decode_individual(individual, feature_cols)
        if not conditions:
            continue
        mask = metrics.evaluate_rule_mask(discretised_df, conditions)
        if float(mask.mean()) >= MIN_COVERAGE:
            survivors += 1
    return survivors

n_naive_survivors = fabricate_rule_pool_naive("baseline", "baseline_pred_class", seed=RNG_SEED)
print(f"naive uniform sampling: {n_naive_survivors}/{N_CANDIDATES} candidates cleared the {MIN_COVERAGE:.0%} coverage floor")
print(f"(with {len(feature_cols)} features, a uniformly-sampled chromosome has ~75% of genes active on average")
print(f" -- a rule with that many ANDed conditions has coverage that collapses toward 0 almost every time)")


naive uniform sampling: 0/400 candidates cleared the 2% coverage floor
(with 58 features, a uniformly-sampled chromosome has ~75% of genes active on average
 -- a rule with that many ANDed conditions has coverage that collapses toward 0 almost every time)


### 4.2 Why this matters beyond just fabrication

This isn't only a fabrication-script problem. `random_gene()` is the literal function
`nsga.py`'s real DEAP toolbox uses to initialise its starting population (`attr_partition`
above) — meaning a real NSGA-II run's **generation 0** faces this exact same near-zero-coverage
problem before any selection pressure has acted. Worth raising with Sakhile directly: if most
of an initial population scores `fidelity=0` under the coverage floor (§4.3's `_score_genotype`
zeroes fidelity below `min_coverage`), NSGA-II's crowding/dominance comparisons have very
little useful signal to select on early, which could slow or even stall convergence depending
on generation budget. Not fixing that here — it's his algorithm's initialisation strategy to
decide on — but the empirical evidence above is worth putting in front of him.

### 4.3 The actual fabrication approach used below

Rather than sampling all `len(feature_cols)` genes uniformly, each candidate chromosome is
built by picking a small number of **active** positions directly (1 to 5, matching realistic
rule lengths seen in the literature this project builds on — Yedjour et al.'s GPRO_RX rules
are similarly short) and leaving every other gene at `-1` (excluded) by construction, rather
than by chance. Everything downstream — decoding, scoring, the coverage filter, the schema
— is unchanged and still the real shared code; only the *initial sampling distribution* is
deliberately sparse instead of uniform.

Pipeline, per condition/extractor pair:

1. Sample a sparse candidate chromosome (1–5 active genes).
2. Decode into `conditions`/`genes` via `common.decode_individual`/`resolve_genes`.
3. Score via `metrics.evaluate_rule_mask`/`fidelity_f1` against the condition's predicted
   classes (`baseline_pred_class` or `sam_pred_class` from notebook 01).
4. Apply the team's minimum-coverage filter (2%, decision C4) — the same degenerate-rule fix
   Sakhile diagnosed and built into `nsga.py`'s `--min-coverage` flag.
5. Sort by fidelity, keep the top candidates as a stand-in "non-dominated pool."

`min_coverage=0.02` is the team's confirmed default (Part F1 of the orientation doc lists it
as fixed at 2%, provisional-but-adopted).


In [6]:
def sample_sparse_chromosome(n_features: int, local_rng: np.random.Generator, max_active: int = 5) -> list:
    '''Build a chromosome with a small number of active genes by construction (see §4.2/4.3
    above for why uniform random_gene() per position doesn't work at this feature count).
    Active positions get a real partition value (0/1/2) via random_gene()'s own value range,
    everything else stays excluded (-1).'''
    n_active = local_rng.integers(1, max_active + 1)
    active_positions = local_rng.choice(n_features, size=n_active, replace=False)
    individual = [-1] * n_features
    for pos in active_positions:
        individual[pos] = int(local_rng.integers(0, 3))  # 0/1/2 = Low/Intermediate/High
    return individual

def fabricate_rule_pool(condition: str, extractor: str, class_col: str, seed: int) -> dict:
    local_rng = np.random.default_rng(seed)
    n_features = len(feature_cols)
    actual_classes = labels_df[class_col]

    candidates = []
    for _ in range(N_CANDIDATES):
        individual = sample_sparse_chromosome(n_features, local_rng)
        length = sum(1 for v in individual if v != -1)
        if length == 0:
            continue  # same empty-rule guard added to nsga.py during the merge review

        conditions = decode_individual(individual, feature_cols)
        mask = metrics.evaluate_rule_mask(discretised_df, conditions)
        coverage = float(mask.mean())
        if coverage < MIN_COVERAGE or mask.sum() == 0:
            continue

        # majority-vote predicted class among covered rows (same convention as nsga.py's
        # _score_genotype), then score fidelity against that
        covered_actual = actual_classes.loc[mask]
        predicted_class = covered_actual.mode().iat[0]
        rule_preds = pd.Series(predicted_class, index=covered_actual.index)
        fidelity = metrics.fidelity_f1(rule_preds, covered_actual)
        rule_length = metrics.comprehensibility({"conditions": conditions})

        candidates.append({
            "conditions": conditions,
            "genes": resolve_genes(conditions, feature_map),
            "predicted_class": predicted_class,
            "length": rule_length,
            "fidelity_f1": round(float(fidelity), 4),
            "coverage": round(coverage, 4),
            "run_ids": [1],  # fabricated — a real run would carry genuine seeded run ids
            "non_dominated": True,  # fabricated stand-in claim, not a real Pareto check — see notebook intro
        })

    candidates.sort(key=lambda r: (-r["fidelity_f1"], r["length"]))
    top = candidates[:TOP_K]
    for i, r in enumerate(top, start=1):
        r["id"] = f"{extractor}_{condition}_{i:04d}"

    return {
        "meta": {
            "extractor": extractor,
            "condition": condition,
            "runs": 1,  # fabricated: a single sampling pass, not the real 30 independent runs
            "config_hash": cfg.config_hash,
        },
        "rules": top,
    }


## 5. Generate and validate all four pools

`ga_baseline` / `ga_sam` stand in for Melisa's branch (not yet started); `nsga_baseline` /
`nsga_sam` stand in for Sakhile's (real algorithm exists in `src/extraction/nsga.py`, just not
run here). `schema.save_rule_pool` validates against the exact same schema Sakhile's real
`nsga.py` writes through — if this cell succeeds, the fabricated pools are genuinely
indistinguishable in *shape* from what the real extractors will eventually produce.


In [7]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

pools = {}
seed_offset = 0
for extractor in ("ga", "nsga"):
    for condition, class_col in (("baseline", "baseline_pred_class"), ("sam", "sam_pred_class")):
        seed_offset += 1
        pool = fabricate_rule_pool(condition, extractor, class_col, seed=RNG_SEED + seed_offset)
        out_path = OUT_DIR / f"{extractor}_{condition}.json"
        schema.save_rule_pool(pool, out_path, feature_map=feature_map)  # validates, then writes
        pools[f"{extractor}_{condition}"] = pool
        print(f"{extractor}_{condition}: {len(pool['rules'])} rules -> {out_path}")


ga_baseline: 15 rules -> ../tests/fixtures/synthetic/rules/ga_baseline.json


ga_sam: 15 rules -> ../tests/fixtures/synthetic/rules/ga_sam.json


nsga_baseline: 15 rules -> ../tests/fixtures/synthetic/rules/nsga_baseline.json


nsga_sam: 15 rules -> ../tests/fixtures/synthetic/rules/nsga_sam.json


## 6. Sanity check against the known ground truth

Notebook 01 planted specific signal genes/pathways (`TRUE_SIGNAL_GENES`/`TRUE_SIGNAL_PATHWAYS`
in the ground-truth metadata). Real data never gives you this — checking it here is only
possible *because* this is synthetic, and it's a genuine test of whether the fidelity-driven
sorting above is doing anything sensible at all, not just producing schema-shaped noise.


In [8]:
true_genes = set(ground_truth["true_signal_genes"].keys())
true_pathways = set(ground_truth["true_signal_pathways"].keys())
true_features = true_genes | true_pathways

for name, pool in pools.items():
    top5_genes = set()
    for r in pool["rules"][:5]:
        top5_genes.update(c["feature"] for c in r["conditions"])
    overlap = top5_genes & true_features
    print(f"{name}: top-5 rules touch {sorted(top5_genes)}")
    print(f"  -> overlap with planted signal ({sorted(true_features)}): {sorted(overlap) or 'none'}")


ga_baseline: top-5 rules touch ['GENE_08', 'GENE_10', 'GENE_12', 'PATHWAY_03', 'PATHWAY_08', 'PATHWAY_25', 'PATHWAY_28', 'PATHWAY_30', 'PATHWAY_32', 'PATHWAY_34']
  -> overlap with planted signal (['GENE_06', 'GENE_11', 'GENE_12', 'GENE_17', 'PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']): ['GENE_12', 'PATHWAY_25', 'PATHWAY_30']
ga_sam: top-5 rules touch ['GENE_13', 'GENE_16', 'GENE_17', 'PATHWAY_03', 'PATHWAY_11', 'PATHWAY_13', 'PATHWAY_16', 'PATHWAY_18', 'PATHWAY_29', 'PATHWAY_30', 'PATHWAY_36', 'PATHWAY_37']
  -> overlap with planted signal (['GENE_06', 'GENE_11', 'GENE_12', 'GENE_17', 'PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']): ['GENE_17', 'PATHWAY_30']
nsga_baseline: top-5 rules touch ['PATHWAY_01', 'PATHWAY_10', 'PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']
  -> overlap with planted signal (['GENE_06', 'GENE_11', 'GENE_12', 'GENE_17', 'PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']): ['PATHWAY_19', 'PATHWAY_25', 'PATHWAY_30']
nsga_sam: top-5 rules touch ['GENE_01', 'GENE_07', 'GENE_12', 'PATHWA

## Next up

With four schema-valid rule pools on disk, the natural next notebook is:

- **`03_promethee_ranking.ipynb`** — a single unified PROMETHEE II run (methodology §4.6.1)
  over all four pools' fidelity / comprehensibility / stability, `pymcdm`-based, producing
  the top-10 ranked rules per condition. `stability()` needs per-subgroup predictions
  (notebook 01's `cancer_type` column) — that wiring happens there, not here.

Once the real pipeline is further along (Angelique's/Bahiya's real checkpoints, Sakhile's
real 30-run NSGA-II output), the fabricated pools here can be swapped for genuinely running
`python -m src.extraction.nsga --condition baseline` (and the SAM equivalent) against this
same synthetic data reshaped into the `artifacts/` layout — worth doing once the pipeline is
stable, to see how much real evolutionary search improves on this notebook's random sampling.
